# Hierarchical Models for Firing Rates & Amplitudes

This notebook loads **wide-format** block-level data (one row per subject, columns = BL + Block_1 … Block_30) and fits **hierarchical linear mixed-effects models** (LMM) with a random intercept per subject.

### Data convention
- **BL** = baseline (pre-stimulation)
- **Odd blocks** (1, 3, 5, …) = stimulation
- **Even blocks** (2, 4, 6, …) = pause

### Models
1. **Block-type effect** – does stimulation change the DV compared to (a) all non-stim epochs, (b) pause-only epochs?
2. **Time effect** – does the DV change across blocks (linear trend)?
3. **Immediate effect** – percent change relative to the previous block.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style='white', context='talk')

print('Setup done.')

## 1. Load & reshape data

In [ ]:
# ── paths ──────────────────────────────────────────────────────
RATE_PATH = r'final_firing_rates_wide.csv'
AMP_PATH  = r'final_amplitudes_wide.csv'


def load_and_melt(path, value_name):
    """
    Load a wide CSV (rows=subjects, cols=BL,Block_1…Block_30)
    and convert it to long format with columns:
        subject_id, block_label, block_number, block_type, value
    """
    df = pd.read_csv(path)
    # identify block columns (everything except Subject_ID)
    block_cols = [c for c in df.columns if c != 'Subject_ID']

    long = df.melt(id_vars='Subject_ID',
                   value_vars=block_cols,
                   var_name='block_label',
                   value_name=value_name)

    # drop missing observations
    long = long.dropna(subset=[value_name]).copy()

    # assign block number (BL → 0, Block_k → k)
    long['block_number'] = long['block_label'].apply(
        lambda x: 0 if x == 'BL' else int(x.split('_')[1])
    )

    # assign block type
    def _type(label):
        if label == 'BL':
            return 'baseline'
        num = int(label.split('_')[1])
        return 'stim' if num % 2 == 1 else 'pause'

    long['block_type'] = long['block_label'].apply(_type)

    # ensure subject is string (for grouping)
    long['Subject_ID'] = long['Subject_ID'].astype(str)

    long = long.sort_values(['Subject_ID', 'block_number']).reset_index(drop=True)
    return long


df_rate = load_and_melt(RATE_PATH, 'firing_rate')
df_amp  = load_and_melt(AMP_PATH,  'amplitude')

print(f'Firing-rate long shape : {df_rate.shape}')
print(f'Amplitude   long shape : {df_amp.shape}')
display(df_rate.head(10))

In [ ]:
# quick sanity: subjects × block types
for label, df in [('Firing Rate', df_rate), ('Amplitude', df_amp)]:
    print(f'\n── {label} ──')
    print(df.groupby('block_type')['Subject_ID'].nunique())
    print(f'Total observations: {len(df)}')

---
## 2. Helper: run & display a MixedLM

In [ ]:
def fit_lmm(formula, data, groups_col='Subject_ID', reml=True):
    """
    Fit a linear mixed-effects model with random intercept per subject.
    Returns the fitted result.
    """
    model = smf.mixedlm(formula, data=data, groups=data[groups_col])
    result = model.fit(reml=reml)
    return result


def print_model(result, title=''):
    """Pretty-print a model summary."""
    print('=' * 70)
    print(title)
    print('=' * 70)
    print(result.summary())
    print()


def p_to_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

---
## 3. Model 1 – Effect of Block Type

### 1A: Stim vs. Non-Stim (baseline + pause combined)
A binary contrast: **stim** blocks vs all **non-stim** epochs (BL + pause).

### 1B: Stim vs. Pause only
Exclude baseline entirely; compare **stim** blocks to **pause** blocks.

In [ ]:
def run_block_type_models(df, dv, label):
    """
    Run block-type models on a long dataframe.
    dv : name of the dependent-variable column
    label : human-readable label (e.g. 'Firing Rate')
    """
    results = {}

    # ── 1A  stim vs non-stim ─────────────────────────────────────
    df_1a = df.copy()
    df_1a['is_stim'] = (df_1a['block_type'] == 'stim').astype(int)

    formula_1a = f'{dv} ~ is_stim'
    res_1a = fit_lmm(formula_1a, df_1a)
    print_model(res_1a,
                f'{label} – Model 1A: Stim vs Non-Stim (BL + Pause)')
    results['1A'] = res_1a

    # ── 1B  stim vs pause only ───────────────────────────────────
    df_1b = df[df['block_type'].isin(['stim', 'pause'])].copy()
    df_1b['is_stim'] = (df_1b['block_type'] == 'stim').astype(int)

    formula_1b = f'{dv} ~ is_stim'
    res_1b = fit_lmm(formula_1b, df_1b)
    print_model(res_1b,
                f'{label} – Model 1B: Stim vs Pause Only')
    results['1B'] = res_1b

    return results

In [ ]:
rate_m1 = run_block_type_models(df_rate, 'firing_rate', 'Firing Rate')

In [ ]:
amp_m1 = run_block_type_models(df_amp, 'amplitude', 'Amplitude')

### Visualisations – Block-Type Effect

In [ ]:
def plot_block_type(df, dv, label, model_result, title='', exclude_baseline=False):
    """Box + strip plot for stim vs non-stim."""
    data = df.copy()
    if exclude_baseline:
        data = data[data['block_type'] != 'baseline']
    
    fig, ax = plt.subplots(figsize=(6, 5))

    palette = {'stim': '#ce84ad', 'pause': '#b0b0b0', 'baseline': '#8db4e2'}
    order = ['baseline', 'stim', 'pause'] if not exclude_baseline else ['stim', 'pause']
    
    sns.boxplot(
        data=data, x='block_type', y=dv, order=order,
        palette=palette, width=0.6, showmeans=True,
        meanprops={'marker': 'D', 'markerfacecolor': 'white',
                   'markeredgecolor': 'black', 'markersize': 7},
        boxprops=dict(alpha=0.5, edgecolor='black'),
        medianprops=dict(color='black', linewidth=1.5),
        showfliers=False, zorder=2, ax=ax
    )
    sns.stripplot(
        data=data, x='block_type', y=dv, order=order,
        palette=palette, size=4, alpha=0.5,
        linewidth=0, zorder=3, ax=ax, jitter=0.2
    )

    # p-value from is_stim coefficient
    p = model_result.pvalues.get('is_stim', np.nan)
    coef = model_result.fe_params.get('is_stim', np.nan)
    ax.set_title(f'{title}\nStim coef = {coef:.3f}, p = {p:.4g} {p_to_stars(p)}',
                 fontsize=13, fontweight='bold')

    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=13, fontweight='bold')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    plt.show()


# Firing Rate
plot_block_type(df_rate, 'firing_rate', 'Firing Rate (Hz)',
                rate_m1['1A'], 'Firing Rate – Stim vs Non-Stim')

plot_block_type(df_rate, 'firing_rate', 'Firing Rate (Hz)',
                rate_m1['1B'], 'Firing Rate – Stim vs Pause',
                exclude_baseline=True)

# Amplitude
plot_block_type(df_amp, 'amplitude', 'Amplitude',
                amp_m1['1A'], 'Amplitude – Stim vs Non-Stim')

plot_block_type(df_amp, 'amplitude', 'Amplitude',
                amp_m1['1B'], 'Amplitude – Stim vs Pause',
                exclude_baseline=True)

---
## 4. Model 2 – Effect of Time

Linear effect of `block_number` on the DV, with optional interaction with `block_type`.

### 2A: Overall time trend (all blocks)
### 2B: Time trend with block-type interaction (stim & pause only)

In [ ]:
def run_time_models(df, dv, label):
    """
    Run time-effect models.
    """
    results = {}

    # ── 2A  overall linear time trend ────────────────────────────
    formula_2a = f'{dv} ~ block_number'
    res_2a = fit_lmm(formula_2a, df)
    print_model(res_2a,
                f'{label} – Model 2A: Linear Time Trend (all blocks)')
    results['2A'] = res_2a

    # ── 2B  time × block_type interaction (stim & pause only) ───
    df_2b = df[df['block_type'].isin(['stim', 'pause'])].copy()
    df_2b['is_stim'] = (df_2b['block_type'] == 'stim').astype(int)

    formula_2b = f'{dv} ~ block_number * is_stim'
    res_2b = fit_lmm(formula_2b, df_2b)
    print_model(res_2b,
                f'{label} – Model 2B: Time × Block-Type Interaction')
    results['2B'] = res_2b

    return results

In [ ]:
rate_m2 = run_time_models(df_rate, 'firing_rate', 'Firing Rate')

In [ ]:
amp_m2 = run_time_models(df_amp, 'amplitude', 'Amplitude')

### Visualisations – Time Effect

In [ ]:
def plot_time_trend(df, dv, label, model_result, title=''):
    """Scatter + regression line per block type across time."""
    data = df.copy()

    palette = {'stim': '#ce84ad', 'pause': '#b0b0b0', 'baseline': '#8db4e2'}

    fig, ax = plt.subplots(figsize=(10, 5))

    for bt in ['baseline', 'stim', 'pause']:
        subset = data[data['block_type'] == bt]
        if subset.empty:
            continue
        ax.scatter(subset['block_number'], subset[dv],
                   c=palette.get(bt, 'gray'), alpha=0.5,
                   label=bt.capitalize(), s=30, edgecolors='none')

    # overall trend line from the model
    x_range = np.linspace(data['block_number'].min(),
                          data['block_number'].max(), 100)
    intercept = model_result.fe_params['Intercept']
    slope = model_result.fe_params.get('block_number', 0)
    ax.plot(x_range, intercept + slope * x_range,
            color='black', linewidth=2, linestyle='--', label='Trend')

    p = model_result.pvalues.get('block_number', np.nan)
    ax.set_title(f'{title}\nSlope = {slope:.4f}, p = {p:.4g} {p_to_stars(p)}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Block Number', fontsize=13)
    ax.set_ylabel(label, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    plt.show()


plot_time_trend(df_rate, 'firing_rate', 'Firing Rate (Hz)',
                rate_m2['2A'], 'Firing Rate – Time Trend')

plot_time_trend(df_amp, 'amplitude', 'Amplitude',
                amp_m2['2A'], 'Amplitude – Time Trend')

---
## 5. Model 3 – Immediate Effect (Percent Change from Previous Block)

For each subject, compute the **percent change** of each block relative to the **preceding** block:

$$\text{pct\_change}_i = \frac{\text{value}_i - \text{value}_{i-1}}{\text{value}_{i-1}} \times 100$$

Then test whether these percent changes differ by block type (stim vs pause).

In [ ]:
def compute_pct_change(df, dv):
    """
    Within each subject, compute % change from the previous block.
    Returns a new dataframe with an extra 'pct_change' column.
    The first block of each subject is dropped (no previous value).
    """
    out = df.sort_values(['Subject_ID', 'block_number']).copy()
    out['prev_value'] = out.groupby('Subject_ID')[dv].shift(1)
    out['pct_change'] = ((out[dv] - out['prev_value']) / out['prev_value']) * 100
    out = out.dropna(subset=['pct_change']).copy()
    # remove infinite values (division by zero if prev_value == 0)
    out = out[np.isfinite(out['pct_change'])].copy()
    return out


df_rate_pct = compute_pct_change(df_rate, 'firing_rate')
df_amp_pct  = compute_pct_change(df_amp,  'amplitude')

print(f'Firing-rate pct-change obs: {len(df_rate_pct)}')
print(f'Amplitude   pct-change obs: {len(df_amp_pct)}')
display(df_rate_pct.head(10))

In [ ]:
def run_immediate_effect_models(df_pct, label):
    """
    Models on the pct_change variable.
    3A: Is % change different from zero overall?
    3B: Does % change differ between stim and pause blocks?
    3C: Does % change differ between stim and pause, with time interaction?
    """
    results = {}

    # ── 3A  overall pct change ───────────────────────────────────
    # intercept-only → tests whether mean pct_change ≠ 0
    res_3a = fit_lmm('pct_change ~ 1', df_pct)
    print_model(res_3a,
                f'{label} – Model 3A: Overall Pct Change ≠ 0?')
    results['3A'] = res_3a

    # ── 3B  stim vs pause (exclude baseline transitions) ────────
    df_3b = df_pct[df_pct['block_type'].isin(['stim', 'pause'])].copy()
    df_3b['is_stim'] = (df_3b['block_type'] == 'stim').astype(int)

    res_3b = fit_lmm('pct_change ~ is_stim', df_3b)
    print_model(res_3b,
                f'{label} – Model 3B: Pct Change Stim vs Pause')
    results['3B'] = res_3b

    # ── 3C  stim vs pause with time interaction ──────────────────
    res_3c = fit_lmm('pct_change ~ is_stim * block_number', df_3b)
    print_model(res_3c,
                f'{label} – Model 3C: Pct Change ~ BlockType × Time')
    results['3C'] = res_3c

    return results

In [ ]:
rate_m3 = run_immediate_effect_models(df_rate_pct, 'Firing Rate')

In [ ]:
amp_m3 = run_immediate_effect_models(df_amp_pct, 'Amplitude')

### Visualisations – Immediate (Percent Change) Effect

In [ ]:
def plot_pct_change(df_pct, label, model_result, title=''):
    """Box + strip plot of pct_change by block type (stim vs pause)."""
    data = df_pct[df_pct['block_type'].isin(['stim', 'pause'])].copy()

    fig, ax = plt.subplots(figsize=(6, 5))
    palette = {'stim': '#ce84ad', 'pause': '#b0b0b0'}

    sns.boxplot(
        data=data, x='block_type', y='pct_change',
        order=['stim', 'pause'], palette=palette,
        width=0.5, showmeans=True,
        meanprops={'marker': 'D', 'markerfacecolor': 'white',
                   'markeredgecolor': 'black', 'markersize': 7},
        boxprops=dict(alpha=0.5, edgecolor='black'),
        medianprops=dict(color='black', linewidth=1.5),
        showfliers=False, zorder=2, ax=ax
    )
    sns.stripplot(
        data=data, x='block_type', y='pct_change',
        order=['stim', 'pause'], palette=palette,
        size=4, alpha=0.5, linewidth=0, zorder=3, ax=ax, jitter=0.2
    )

    ax.axhline(0, color='gray', linewidth=1, linestyle='--', zorder=1)

    p = model_result.pvalues.get('is_stim', np.nan)
    coef = model_result.fe_params.get('is_stim', np.nan)
    ax.set_title(f'{title}\nStim coef = {coef:.2f}%, p = {p:.4g} {p_to_stars(p)}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(['Stim', 'Pause'], fontsize=13, fontweight='bold')
    ax.set_ylabel(f'% Change from Previous Block\n({label})',
                  fontsize=12, fontweight='bold')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    plt.show()


plot_pct_change(df_rate_pct, 'Firing Rate',
                rate_m3['3B'], 'Firing Rate – Immediate Effect')

plot_pct_change(df_amp_pct, 'Amplitude',
                amp_m3['3B'], 'Amplitude – Immediate Effect')

In [ ]:
def plot_pct_over_time(df_pct, label, title=''):
    """Pct change scatter across block number, colored by block type."""
    data = df_pct[df_pct['block_type'].isin(['stim', 'pause'])].copy()

    palette = {'stim': '#ce84ad', 'pause': '#b0b0b0'}

    fig, ax = plt.subplots(figsize=(10, 5))
    for bt in ['stim', 'pause']:
        sub = data[data['block_type'] == bt]
        ax.scatter(sub['block_number'], sub['pct_change'],
                   c=palette[bt], label=bt.capitalize(),
                   alpha=0.5, s=30, edgecolors='none')

    # per-type mean at each block number
    for bt, color in palette.items():
        means = data[data['block_type'] == bt].groupby('block_number')['pct_change'].mean()
        ax.plot(means.index, means.values, color=color,
                linewidth=2, marker='o', markersize=5)

    ax.axhline(0, color='gray', linewidth=1, linestyle='--', zorder=1)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Block Number', fontsize=13)
    ax.set_ylabel(f'% Change ({label})', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    plt.tight_layout()
    plt.show()


plot_pct_over_time(df_rate_pct, 'Firing Rate',
                   'Firing Rate – Pct Change Over Time')

plot_pct_over_time(df_amp_pct, 'Amplitude',
                   'Amplitude – Pct Change Over Time')

---
## 6. Summary Table

In [ ]:
def extract_summary_row(model_id, measure, result, key_coef):
    """Extract one row for the summary table."""
    coef = result.fe_params.get(key_coef, np.nan)
    p = result.pvalues.get(key_coef, np.nan)
    ci = result.conf_int().loc[key_coef] if key_coef in result.conf_int().index else [np.nan, np.nan]
    return {
        'Model': model_id,
        'Measure': measure,
        'Coefficient': key_coef,
        'Estimate': round(coef, 4),
        'p-value': round(p, 4),
        'Sig.': p_to_stars(p),
        'CI_low': round(ci[0], 4) if not np.isnan(ci[0]) else np.nan,
        'CI_high': round(ci[1], 4) if not np.isnan(ci[1]) else np.nan,
        'N_obs': int(result.nobs),
    }

rows = []

# Model 1
rows.append(extract_summary_row('1A: Stim vs Non-Stim', 'Rate', rate_m1['1A'], 'is_stim'))
rows.append(extract_summary_row('1A: Stim vs Non-Stim', 'Amp',  amp_m1['1A'],  'is_stim'))
rows.append(extract_summary_row('1B: Stim vs Pause',    'Rate', rate_m1['1B'], 'is_stim'))
rows.append(extract_summary_row('1B: Stim vs Pause',    'Amp',  amp_m1['1B'],  'is_stim'))

# Model 2
rows.append(extract_summary_row('2A: Time Trend',       'Rate', rate_m2['2A'], 'block_number'))
rows.append(extract_summary_row('2A: Time Trend',       'Amp',  amp_m2['2A'],  'block_number'))
rows.append(extract_summary_row('2B: Time × Type',      'Rate', rate_m2['2B'], 'block_number:is_stim'))
rows.append(extract_summary_row('2B: Time × Type',      'Amp',  amp_m2['2B'],  'block_number:is_stim'))

# Model 3
rows.append(extract_summary_row('3A: Pct Change ≠ 0',   'Rate', rate_m3['3A'], 'Intercept'))
rows.append(extract_summary_row('3A: Pct Change ≠ 0',   'Amp',  amp_m3['3A'],  'Intercept'))
rows.append(extract_summary_row('3B: Pct Stim vs Pause','Rate', rate_m3['3B'], 'is_stim'))
rows.append(extract_summary_row('3B: Pct Stim vs Pause','Amp',  amp_m3['3B'],  'is_stim'))

df_summary = pd.DataFrame(rows)
display(df_summary)

In [ ]:
# Save summary to CSV
df_summary.to_csv('hierarchical_models_summary.csv', index=False)
print('Summary saved to hierarchical_models_summary.csv')